# 4. Feature Engineering

Notebook นี้นิยาม preprocessing เป็น `ColumnTransformer` เพื่อให้ทุกขั้นตอนถูก fit ภายใน model `Pipeline` แทนการ transform แล้วบันทึกข้อมูลล่วงหน้า ซึ่งสำคัญมากตอน cross-validation

In [8]:
import pandas as pd

train_df = pd.read_parquet("../data/processed/train_pre_impute.parquet")
valid_df = pd.read_parquet("../data/processed/valid_pre_impute.parquet")
test_df = pd.read_parquet("../data/processed/test_pre_impute.parquet")

print(train_df.shape, valid_df.shape, test_df.shape)

(560, 18) (120, 18) (120, 18)


In [9]:
X_train = train_df.drop(columns="Dropout_Risk")
y_train = train_df["Dropout_Risk"]

X_train.isnull().sum()

Age                        0
Year_of_Study              0
Department                13
Residence_Type            12
Attendance_Percent        36
Study_Hours_Per_Day       24
Previous_GPA              26
Backlogs                  11
Screen_Time_Hours         22
Part_Time_Job             16
Family_Income_Bracket      7
Financial_Stress_Score    30
Family_Support_Score      33
Stress_Level              35
Anxiety_Score             28
Motivation_Score          32
Counseling_Access          0
dtype: int64

In [10]:
missing_numeric_cols = [
    "Attendance_Percent",
    "Study_Hours_Per_Day",
    "Previous_GPA",
    "Backlogs",
    "Screen_Time_Hours",
    "Financial_Stress_Score",
    "Family_Support_Score",
    "Stress_Level",
    "Anxiety_Score",
    "Motivation_Score",
]

X_train[missing_numeric_cols].skew().sort_values()

Family_Support_Score     -0.245835
Attendance_Percent       -0.245026
Stress_Level             -0.077685
Anxiety_Score            -0.034004
Financial_Stress_Score   -0.012787
Previous_GPA             -0.000954
Screen_Time_Hours         0.004666
Motivation_Score          0.099014
Study_Hours_Per_Day       0.161590
Backlogs                  1.470276
dtype: float64

## Preprocessing design

- ใช้ **17 raw input features** หลัง feature reduction
- Numeric features ใช้ `SimpleImputer(mean)` และ `StandardScaler`
- `Backlogs` ใช้ median imputation ตามแนวทางเดิม แล้ว scale
- `Department`, `Residence_Type`, `Part_Time_Job` ใช้ constant imputation + `OneHotEncoder(handle_unknown="ignore")`
- `Family_Income_Bracket` ใช้ ordinal order: Low → Lower-Middle → Middle → Upper-Middle → High โดย missing/unknown จะถูก encode เป็น `-1`
- `Counseling_Access` encode No=0, Yes=1

Preprocessing ทั้งหมดถูกรวมไว้ใน `ColumnTransformer` เพื่อเอาไปใช้เป็นขั้นแรกของ model `Pipeline`.


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

backlog_cols = ["Backlogs"]
unordered_cols = ["Department", "Residence_Type", "Part_Time_Job"]
income_cols = ["Family_Income_Bracket"]
counseling_cols = ["Counseling_Access"]

# Numeric columns อื่นทั้งหมด รวมคอลัมน์ที่ไม่มี missing ด้วย เพื่อให้ pipeline รองรับ raw input ได้ครบ
numeric_cols = [
    col for col in X_train.select_dtypes(include="number").columns
    if col not in backlog_cols
]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
])

backlog_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

unordered_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

income_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OrdinalEncoder(
        categories=[["Low", "Lower-Middle", "Middle", "Upper-Middle", "High"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )),
    ("scaler", StandardScaler()),
])

counseling_pipe = Pipeline([
    ("encoder", OrdinalEncoder(
        categories=[["No", "Yes"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipe, numeric_cols),
        ("backlogs", backlog_pipe, backlog_cols),
        ("unordered", unordered_pipe, unordered_cols),
        ("income", income_pipe, income_cols),
        ("counseling", counseling_pipe, counseling_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

print("Numeric:", numeric_cols)
print("Unordered categorical:", unordered_cols)

Numeric: ['Age', 'Year_of_Study', 'Attendance_Percent', 'Study_Hours_Per_Day', 'Previous_GPA', 'Screen_Time_Hours', 'Financial_Stress_Score', 'Family_Support_Score', 'Stress_Level', 'Anxiety_Score', 'Motivation_Score']
Unordered categorical: ['Department', 'Residence_Type', 'Part_Time_Job']


In [12]:
# ทดลอง fit preprocessing กับ training data เพื่อเช็กผลลัพธ์
X_train_transformed = preprocessor.fit_transform(X_train)

print("Raw feature shape        :", X_train.shape)
print("Transformed feature shape:", X_train_transformed.shape)


Raw feature shape        : (560, 17)
Transformed feature shape: (560, 28)


In [13]:
import numpy as np

print("NaN after preprocessing:", np.isnan(X_train_transformed).sum())

NaN after preprocessing: 0


In [14]:
from pathlib import Path
import joblib

Path("../models/preprocessing").mkdir(parents=True, exist_ok=True)
joblib.dump(preprocessor, "../models/preprocessing/preprocessor.joblib")
print("Saved preprocessor")


Saved preprocessor
